# **Modelos Predictivos Saber 11 - Departamento de Caldas**

Este notebook implementa el proceso de selección, limpieza, alistamiento y análisis exploratorio de los datos de las pruebas Saber 11 para el departamento de Caldas, como parte del Proyecto 2 del curso *Analítica Computacional para la Toma de Decisiones*. El producto final está orientado al **Ministerio de Educación** como usuario final, y busca responder tres preguntas de negocio relacionadas con equidad socioeconómica, desempeño territorial y brechas de género.

## Tarea 4 - Modelamiento

Se exploran diferentes configuraciones de modelo, se realiza ingeniería de características, se emplean diferentes métodos de estimación, y se comparan y seleccionan las mejores alternativas.

---

Daniel Benavides - 202220428 

Juanita Cortés - 202222129 

Andrés Felipe Herrera - 202220888

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import altair as alt
from prettytable import PrettyTable

from matplotlib import font_manager
plt.rcParams['font.family'] = 'Arial'

alt.data_transformers.enable('vegafusion')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras import layers, callbacks

import mlflow
mlflow.set_tracking_uri("mlruns")

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## Carga de datos

In [ ]:
df = pd.read_csv('../data/saber11_features.csv')

print(f"Filas: {len(df)} | Columnas: {df.shape[1]}")
df.head(5)

## 1. Pregunta de negocio

### *¿Cuál es el puntaje global esperado para un estudiante de Caldas dado su perfil socioeconómico y el tipo de institución educativa?*

Esta pregunta se responde con un **modelo de regresión**, ya que la variable objetivo es continua:

$$
Y = punt\_global
$$

El objetivo es estimar el puntaje global esperado de un estudiante del departamento de Caldas a partir de variables asociadas a su contexto socioeconómico y a las características de la institución educativa. Esta predicción puede apoyar la identificación de perfiles con menor desempeño esperado y orientar estrategias de acompañamiento académico.

## 2. Pregunta de negocio

### *¿Puede identificarse si un estudiante está en riesgo de obtener un puntaje global por debajo del umbral de bajo desempeño, según sus características socioeconómicas y escolares?*

Esta pregunta se responde con un **modelo de clasificación binaria**. La variable objetivo es:

$$\text{bajo\_rendimiento} = \begin{cases} 1 & \text{si } \texttt{punt\_global} < 230 \\ 0 & \text{en otro caso} \end{cases}$$

El umbral de 230 puntos delimita el tercio inferior de la distribución de puntajes en Caldas (~33.6% de los estudiantes), siendo operacionalmente accionable para el Ministerio: identifica una minoría prioritaria sin el severo desbalance de clases que produciría un umbral más bajo.

Se evalúan cinco configuraciones que responden dos preguntas de diseño: (1) ¿aportan los cuatro activos del hogar individualmente más que su índice resumen?, y (2) ¿qué profundidad y ancho captura mejor las interacciones entre features socioeconómicos y escolares?

### 2.1 Feature Engineering

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay)
from sklearn.utils.class_weight import compute_class_weight

df_p2 = df.dropna(subset=['bajo_rendimiento']).copy()
TARGET_P2 = 'bajo_rendimiento'

FEATURES_REDUCIDOS = [
    'estrato_num', 'edu_madre_num', 'edu_padre_num',
    'indice_activos', 'es_privado', 'es_urbano', 'genero_num'
]
FEATURES_COMPLETOS = [
    'estrato_num', 'edu_madre_num', 'edu_padre_num',
    'indice_activos',
    'fami_tienecomputador', 'fami_tieneinternet',
    'fami_tieneautomovil', 'fami_tienelavadora',
    'es_privado', 'es_urbano', 'genero_num'
]

df_model_p2 = df_p2[FEATURES_COMPLETOS + [TARGET_P2]].dropna()
print(f"Shape: {df_model_p2.shape}")
print(f"Balance: {df_model_p2[TARGET_P2].value_counts(normalize=True).round(3).to_dict()}")

### 2.2 Preprocesamiento

In [ ]:
def preparar_datos(features):
    X = df_model_p2[features].values
    y = df_model_p2[TARGET_P2].values

    X = SimpleImputer(strategy='median').fit_transform(X)
    X = StandardScaler().fit_transform(X)

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )
    X_tr, X_va, y_tr, y_va = train_test_split(
        X_tr, y_tr, test_size=0.2, random_state=SEED, stratify=y_tr
    )
    return X_tr, X_va, X_te, y_tr, y_va, y_te

X2_train, X2_val, X2_test, y2_train, y2_val, y2_test = preparar_datos(FEATURES_COMPLETOS)

clases = np.unique(y2_train)
pesos  = compute_class_weight('balanced', classes=clases, y=y2_train)
class_weight_p2 = dict(zip(clases, pesos))

print(f"Train: {X2_train.shape} | Val: {X2_val.shape} | Test: {X2_test.shape}")
print(f"Class weights: {class_weight_p2}")

### 2.3 Experimentos

In [ ]:
experimentos_p2 = [
    {
        "nombre": "baseline_reducido",
        "features": FEATURES_REDUCIDOS,
        "capas": [16, 8],
        "dropout": 0.0,
        "learning_rate": 0.001,
        "epochs": 100,
        "batch_size": 64,
        "descripcion": "Sin assets individuales, sin regularizacion. Referencia minima."
    },
    {
        "nombre": "regularizado_reducido",
        "features": FEATURES_REDUCIDOS,
        "capas": [32, 16],
        "dropout": 0.3,
        "learning_rate": 0.001,
        "epochs": 100,
        "batch_size": 64,
        "descripcion": "Sin assets individuales, con dropout. Evalua regularizacion en features reducidos."
    },
    {
        "nombre": "completo_medio",
        "features": FEATURES_COMPLETOS,
        "capas": [64, 32],
        "dropout": 0.3,
        "learning_rate": 0.001,
        "epochs": 100,
        "batch_size": 64,
        "descripcion": "Con assets individuales. Evalua si aportan sobre el indice resumen."
    },
    {
        "nombre": "completo_profundo",
        "features": FEATURES_COMPLETOS,
        "capas": [128, 64, 32],
        "dropout": 0.3,
        "learning_rate": 0.0005,
        "epochs": 100,
        "batch_size": 64,
        "descripcion": "Mas capas para capturar interacciones entre activos y estrato."
    },
    {
        "nombre": "completo_ancho",
        "features": FEATURES_COMPLETOS,
        "capas": [256, 64],
        "dropout": 0.4,
        "learning_rate": 0.0005,
        "epochs": 100,
        "batch_size": 128,
        "descripcion": "Primera capa muy ancha para capturar combinaciones parciales de activos."
    }
]

### 2.4 Entrenamiento con MLflow

In [ ]:
def construir_modelo(n_inputs, capas, dropout, learning_rate):
    seq = [layers.Input(shape=(n_inputs,))]
    for unidades in capas:
        seq.append(layers.Dense(unidades, activation='relu'))
        if dropout > 0:
            seq.append(layers.Dropout(dropout))
    seq.append(layers.Dense(1, activation='sigmoid'))

    m = tf.keras.Sequential(seq)
    m.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    return m

mlflow.set_experiment("P2_clasificacion_riesgo")

resultados_p2 = {}

for config in experimentos_p2:
    nombre = config["nombre"]

    X_tr, X_va, X_te, y_tr, y_va, y_te = preparar_datos(config["features"])

    with mlflow.start_run(run_name=nombre):
        mlflow.log_param("capas",         config["capas"])
        mlflow.log_param("dropout",       config["dropout"])
        mlflow.log_param("learning_rate", config["learning_rate"])
        mlflow.log_param("batch_size",    config["batch_size"])
        mlflow.log_param("n_features",    len(config["features"]))
        mlflow.log_param("descripcion",   config["descripcion"])

        modelo = construir_modelo(
            n_inputs=len(config["features"]),
            capas=config["capas"],
            dropout=config["dropout"],
            learning_rate=config["learning_rate"]
        )

        early_stop_p2 = callbacks.EarlyStopping(
            monitor='val_auc', patience=10,
            restore_best_weights=True, mode='max'
        )

        history = modelo.fit(
            X_tr, y_tr,
            validation_data=(X_va, y_va),
            epochs=config["epochs"],
            batch_size=config["batch_size"],
            class_weight=class_weight_p2,
            callbacks=[early_stop_p2],
            verbose=0
        )

        y_prob = modelo.predict(X_te, verbose=0).flatten()
        y_pred = (y_prob >= 0.5).astype(int)

        acc  = accuracy_score(y_te, y_pred)
        prec = precision_score(y_te, y_pred)
        rec  = recall_score(y_te, y_pred)
        f1   = f1_score(y_te, y_pred)
        auc  = roc_auc_score(y_te, y_prob)

        mlflow.log_metric("accuracy",  acc)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("recall",    rec)
        mlflow.log_metric("f1",        f1)
        mlflow.log_metric("auc_roc",   auc)
        mlflow.log_metric("epochs_reales", len(history.history['loss']))

        modelo.save(f"model_p2_{nombre}.keras")

        resultados_p2[nombre] = {
            "acc": acc, "prec": prec, "rec": rec, "f1": f1, "auc": auc,
            "history": history, "y_prob": y_prob, "y_pred": y_pred, "y_te": y_te
        }
        print(f"[{nombre}]  Acc={acc:.3f}  Prec={prec:.3f}  Rec={rec:.3f}  F1={f1:.3f}  AUC={auc:.3f}")

### 2.5 Comparación de modelos

In [ ]:
tabla_p2 = PrettyTable()
tabla_p2.field_names = ["Experimento", "Accuracy", "Precision", "Recall", "F1", "AUC-ROC"]
for nombre, res in resultados_p2.items():
    tabla_p2.add_row([
        nombre,
        f"{res['acc']:.3f}", f"{res['prec']:.3f}",
        f"{res['rec']:.3f}", f"{res['f1']:.3f}", f"{res['auc']:.3f}"
    ])
print(tabla_p2)

mejor_p2 = max(resultados_p2, key=lambda n: resultados_p2[n]['f1'])
print(f"\nMejor modelo por F1: {mejor_p2}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for nombre, res in resultados_p2.items():
    axes[0].plot(res['history'].history['val_loss'], label=nombre)
axes[0].set_title("Val Loss por experimento")
axes[0].set_xlabel("Época")
axes[0].legend(fontsize=7)

for nombre, res in resultados_p2.items():
    axes[1].plot(res['history'].history['val_auc'], label=nombre)
axes[1].set_title("Val AUC por experimento")
axes[1].set_xlabel("Época")
axes[1].legend(fontsize=7)

cm = confusion_matrix(resultados_p2[mejor_p2]['y_te'], resultados_p2[mejor_p2]['y_pred'])
ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Bajo rend.']).plot(ax=axes[2], colorbar=False)
axes[2].set_title(f"Mejor modelo: {mejor_p2}\nF1={resultados_p2[mejor_p2]['f1']:.3f}  AUC={resultados_p2[mejor_p2]['auc']:.3f}")

plt.tight_layout()
plt.show()

## 3. Pregunta de negocio

### *¿Puede predecirse el nivel de desempeño en inglés de un estudiante (A−, A1, A2, B1, B+) a partir de su perfil académico, socioeconómico y de género?*

Esta pregunta se responde con un modelo de clasificación multiclase, ya que la variable objetivo es categórica:


Y = desemp_ingles \in {A-, A1, A2, B1, B+}


El objetivo es predecir el nivel de desempeño en inglés de un estudiante de Caldas a partir de su perfil académico, socioeconómico y de género. Esta predicción puede apoyar la identificación de perfiles con bajo desempeño esperado y orientar estrategias de refuerzo en competencias de inglés.


### 3.1 Feature Engineering

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
from sklearn.utils.class_weight import compute_class_weight

df_p3 = df.dropna(subset=['desemp_ingles']).copy()
TARGET_P3 = 'desemp_ingles'

# Mantener únicamente las categorías esperadas
niveles_ingles = ['A-', 'A1', 'A2', 'B1', 'B+']
df_p3 = df_p3[df_p3[TARGET_P3].isin(niveles_ingles)].copy()

# Codificar target multiclase
label_encoder_p3 = LabelEncoder()
df_p3['desemp_ingles_encoded'] = label_encoder_p3.fit_transform(df_p3[TARGET_P3])

TARGET_P3_ENCODED = 'desemp_ingles_encoded'

FEATURES_REDUCIDOS_P3 = [
    'punt_matematicas',
    'punt_lectura_critica',
    'punt_c_naturales',
    'punt_sociales_ciudadanas',
    'estrato_num',
    'edu_madre_num',
    'edu_padre_num',
    'indice_activos',
    'genero_num'
]

FEATURES_COMPLETOS_P3 = [
    'punt_matematicas',
    'punt_lectura_critica',
    'punt_c_naturales',
    'punt_sociales_ciudadanas',
    'estrato_num',
    'edu_madre_num',
    'edu_padre_num',
    'indice_activos',
    'fami_tienecomputador',
    'fami_tieneinternet',
    'fami_tieneautomovil',
    'fami_tienelavadora',
    'es_privado',
    'es_urbano',
    'cole_bilingue',
    'genero_num'
]

df_model_p3 = df_p3[FEATURES_COMPLETOS_P3 + [TARGET_P3, TARGET_P3_ENCODED]].dropna().copy()

print(f"Shape: {df_model_p3.shape}")
print("Clases:", list(label_encoder_p3.classes_))
print("Distribución del target:")
print(df_model_p3[TARGET_P3].value_counts(normalize=True).round(3))

### 3.2 Preprocesamiento

In [ ]:
def preparar_datos_p3(features):
    X = df_model_p3[features].values
    y = df_model_p3[TARGET_P3_ENCODED].values

    X = SimpleImputer(strategy='median').fit_transform(X)
    X = StandardScaler().fit_transform(X)

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y,
        test_size=0.2,
        random_state=SEED,
        stratify=y
    )

    X_tr, X_va, y_tr, y_va = train_test_split(
        X_tr, y_tr,
        test_size=0.2,
        random_state=SEED,
        stratify=y_tr
    )

    return X_tr, X_va, X_te, y_tr, y_va, y_te

X3_train, X3_val, X3_test, y3_train, y3_val, y3_test = preparar_datos_p3(FEATURES_COMPLETOS_P3)

clases_p3 = np.unique(y3_train)
pesos_p3 = compute_class_weight(
    class_weight='balanced',
    classes=clases_p3,
    y=y3_train
)

class_weight_p3 = dict(zip(clases_p3, pesos_p3))

print(f"Train: {X3_train.shape} | Val: {X3_val.shape} | Test: {X3_test.shape}")
print(f"Class weights: {class_weight_p3}")

### 3.3 Experimentos

In [ ]:
experimentos_p3 = [
    {
        "nombre": "baseline_reducido",
        "features": FEATURES_REDUCIDOS_P3,
        "capas": [32, 16],
        "dropout": 0.0,
        "learning_rate": 0.001,
        "epochs": 100,
        "batch_size": 64,
        "descripcion": "Features academicos, socioeconomicos y genero. Sin regularizacion."
    },
    {
        "nombre": "regularizado_reducido",
        "features": FEATURES_REDUCIDOS_P3,
        "capas": [64, 32],
        "dropout": 0.3,
        "learning_rate": 0.001,
        "epochs": 100,
        "batch_size": 64,
        "descripcion": "Features reducidos con dropout para controlar sobreajuste."
    },
    {
        "nombre": "completo_medio",
        "features": FEATURES_COMPLETOS_P3,
        "capas": [64, 32],
        "dropout": 0.3,
        "learning_rate": 0.001,
        "epochs": 100,
        "batch_size": 64,
        "descripcion": "Incluye activos individuales y variables escolares."
    },
    {
        "nombre": "completo_profundo",
        "features": FEATURES_COMPLETOS_P3,
        "capas": [128, 64, 32],
        "dropout": 0.3,
        "learning_rate": 0.0005,
        "epochs": 100,
        "batch_size": 64,
        "descripcion": "Mayor profundidad para capturar interacciones entre rendimiento academico y contexto."
    },
    {
        "nombre": "completo_ancho",
        "features": FEATURES_COMPLETOS_P3,
        "capas": [256, 128],
        "dropout": 0.4,
        "learning_rate": 0.0005,
        "epochs": 100,
        "batch_size": 128,
        "descripcion": "Arquitectura mas ancha para capturar combinaciones no lineales entre variables."
    }
]